We're building a custom dataset by combining ETF options chains from yfinance with earnings dates for each ETF's top holdings, scraped from stockanalysis.com. We verified scraping is allowed via their robots.txt. Our main question is whether implied volatility in ETF options responds to earnings events from the ETF's largest individual holdings

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
import json

Use yfinance api and scrape stock analysis

In [ ]:
#Yfinance API
import yfinance as yf

#scrape data from stockanalysis.com
url = "https://stockanalysis.com/stocks/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', {'class': 'stock-table'})

Join yfinance options and history and stock analysis earnings for each stock in ETF

# Section 1 — Price History

In [2]:
# Imports

import pandas as pd
from data_collection import get_price_history

In [3]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]
PERIOD  = "6mo"
OUTPUT_CSV = "price_history.csv"

In [4]:
# Fetch price history for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching price history: {ticker} ...", end=" ")
    try:
        df = get_price_history(ticker, period=PERIOD)
        frames.append(df)
        print(f"✓  {len(df):,} rows")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

prices = pd.concat(frames, ignore_index=False)

Fetching price history: VOO ... 

INFO | get_price_history('VOO', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.
INFO | get_price_history('QQQ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows
Fetching price history: QQQ ... ✓  125 rows
Fetching price history: ARKQ ... 

INFO | get_price_history('ARKQ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.
INFO | get_price_history('BOTZ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows
Fetching price history: BOTZ ... ✓  125 rows


In [5]:
# Clean and standardize

# Reset index so Date becomes a regular column
prices = prices.reset_index()

# Rename to match acceptance criteria exactly
prices = prices.rename(columns={"index": "Date", "ticker": "ticker"})

# Keep only required columns
KEEP_COLS = ["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
cols = [c for c in KEEP_COLS if c in prices.columns]
prices = prices[cols]

# Ensure correct dtypes
prices["Date"]   = pd.to_datetime(prices["Date"]).dt.tz_localize(None)
prices["Open"]   = pd.to_numeric(prices["Open"],   errors="coerce")
prices["High"]   = pd.to_numeric(prices["High"],   errors="coerce")
prices["Low"]    = pd.to_numeric(prices["Low"],    errors="coerce")
prices["Close"]  = pd.to_numeric(prices["Close"],  errors="coerce")
prices["Volume"] = pd.to_numeric(prices["Volume"], errors="coerce")

# Sort for readability
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

prices.head(10)


Price,ticker,Date,Open,High,Low,Close,Volume
0,ARKQ,2025-10-01,110.815463,113.388857,110.715721,113.229263,262900
1,ARKQ,2025-10-02,115.174275,115.503432,114.176837,115.303940,297500
2,ARKQ,2025-10-03,116.361226,116.999586,114.605731,116.321327,445000
3,ARKQ,2025-10-06,120.051745,121.976802,119.782432,121.737419,704100
4,ARKQ,2025-10-07,121.996748,123.192682,119.263768,120.201363,708300
5,ARKQ,2025-10-08,120.271184,122.744828,119.847270,122.665039,349700
6,ARKQ,2025-10-09,122.505443,123.450016,120.131541,121.398285,304800
7,ARKQ,2025-10-10,121.498031,122.914400,115.483475,115.653038,626600
8,ARKQ,2025-10-13,119.124128,120.510567,117.967095,119.842285,360000
9,ARKQ,2025-10-14,117.588069,121.936903,115.433605,120.420799,319100


In [6]:
# Verify no major gaps in the time series

print("=== Gap Analysis ===\n")

for ticker in TICKERS:
    subset = prices[prices["ticker"] == ticker].copy()

    if subset.empty:
        print(f"{ticker}: No data.\n")
        continue

    # Expected trading days (roughly 5 per week, minus holidays)
    date_range = pd.bdate_range(subset["Date"].min(), subset["Date"].max())
    expected   = len(date_range)
    actual     = len(subset)
    missing    = expected - actual

    # Find specific gap dates
    actual_dates   = set(subset["Date"].dt.date)
    expected_dates = set(date_range.date)
    gap_dates      = sorted(expected_dates - actual_dates)

    print(f"{ticker}:")
    print(f"  Date range : {subset['Date'].min().date()} → {subset['Date'].max().date()}")
    print(f"  Expected   : {expected} trading days")
    print(f"  Actual     : {actual} rows")
    print(f"  Missing    : {missing} days", end="")

    if missing == 0:
        print(" ✓ No gaps")
    elif missing <= 5:
        print(f" — likely holidays: {gap_dates}")
    else:
        print(f" ⚠ Potential data issue — gap dates: {gap_dates[:10]}")
    print()

=== Gap Analysis ===

VOO:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

QQQ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

ARKQ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

BOTZ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — 

In [7]:
# Quick summary stats

summary = (
    prices.groupby("ticker")
    .agg(
        rows        = ("Close", "count"),
        start_date  = ("Date",  "min"),
        end_date    = ("Date",  "max"),
        avg_close   = ("Close", "mean"),
        min_close   = ("Close", "min"),
        max_close   = ("Close", "max"),
        avg_volume  = ("Volume","mean"),
    )
    .round(2)
)
print(summary)


        rows start_date   end_date  avg_close  min_close  max_close  \
ticker                                                                
ARKQ     125 2025-10-01 2026-03-31     118.94     101.00     134.05   
BOTZ     125 2025-10-01 2026-03-31      36.52      31.99      39.66   
QQQ      125 2025-10-01 2026-03-31     609.55     558.28     634.15   
VOO      125 2025-10-01 2026-03-31     620.67     580.93     637.69   

        avg_volume  
ticker              
ARKQ      291056.8  
BOTZ      878903.2  
QQQ     61108528.8  
VOO      9872143.2  


/var/folders/v1/_0s9cgvs7f50tdjbmzh49yn00000gn/T/ipykernel_32061/1166243420.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


In [8]:
# Save to CSV

prices.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV → {OUTPUT_CSV}  ({prices.shape[0]:,} rows, {prices.shape[1]} columns)")



Saved CSV → price_history.csv  (500 rows, 7 columns)


In [9]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV, parse_dates=["Date"])
assert reloaded.shape == prices.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, {reloaded.shape[1]} columns.")
reloaded.head()

Reload check passed — 500 rows, 7 columns.


,ticker,Date,Open,High,Low,Close,Volume
0,ARKQ,2025-10-01,110.815463,113.388857,110.715721,113.229263,262900
1,ARKQ,2025-10-02,115.174275,115.503432,114.176837,115.303940,297500
2,ARKQ,2025-10-03,116.361226,116.999586,114.605731,116.321327,445000
3,ARKQ,2025-10-06,120.051745,121.976802,119.782432,121.737419,704100
4,ARKQ,2025-10-07,121.996748,123.192682,119.263768,120.201363,708300


# Section 2 — Options Chain

In [10]:
# Imports 

import pandas as pd
from data_collection import get_options_chain

In [11]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]

KEEP_COLS = [
    "ticker",
    "expiration",
    "contractType",   # renamed from option_type for clarity
    "strike",
    "bid",
    "ask",
    "volume",
    "openInterest",
    "impliedVolatility",
]

OUTPUT_CSV     = "options_chain.csv"
OUTPUT_PARQUET = "options_chain.parquet"

In [12]:
# Fetch options chain for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching options chain: {ticker} ...", end=" ")
    try:
        df = get_options_chain(ticker)

        # Rename option_type -> contractType to match acceptance criteria
        df = df.rename(columns={"option_type": "contractType"})

        # Keep only the required columns (drop any that don't exist gracefully)
        cols = [c for c in KEEP_COLS if c in df.columns]
        df = df[cols]

        frames.append(df)
        print(f"✓  {len(df):,} rows")

    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

options = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows collected: {len(options):,}")

Fetching options chain: VOO ... 

INFO | get_options_chain('VOO'): 1654 rows across 16 expirations.


✓  1,654 rows
Fetching options chain: QQQ ... 

INFO | get_options_chain('QQQ'): 7170 rows across 30 expirations.


✓  7,170 rows
Fetching options chain: ARKQ ... 

INFO | get_options_chain('ARKQ'): 225 rows across 5 expirations.


✓  225 rows
Fetching options chain: BOTZ ... 

INFO | get_options_chain('BOTZ'): 96 rows across 4 expirations.


✓  96 rows

Total rows collected: 9,145


In [13]:
# Light cleaning

# Ensure correct dtypes
options["expiration"]       = pd.to_datetime(options["expiration"])
options["strike"]           = pd.to_numeric(options["strike"],           errors="coerce")
options["bid"]              = pd.to_numeric(options["bid"],              errors="coerce")
options["ask"]              = pd.to_numeric(options["ask"],              errors="coerce")
options["volume"]           = pd.to_numeric(options["volume"],           errors="coerce")
options["openInterest"]     = pd.to_numeric(options["openInterest"],     errors="coerce")
options["impliedVolatility"]= pd.to_numeric(options["impliedVolatility"],errors="coerce")

# Drop rows where strike or IV is missing — not useful for analysis
before = len(options)
options = options.dropna(subset=["strike", "impliedVolatility"])
dropped = before - len(options)
if dropped:
    print(f"Dropped {dropped:,} rows with missing strike or impliedVolatility.")

# Sort for readability
options = options.sort_values(
    ["ticker", "expiration", "contractType", "strike"]
).reset_index(drop=True)

options.head(10)


,ticker,expiration,contractType,strike,bid,ask,volume,openInterest,impliedVolatility
0,ARKQ,2026-04-17,call,105.0,7.30,9.70,16.0,1,0.549321
1,ARKQ,2026-04-17,call,110.0,3.60,5.50,12.0,6,0.434576
2,ARKQ,2026-04-17,call,115.0,1.65,2.65,5.0,3,0.387091
3,ARKQ,2026-04-17,call,116.0,1.35,2.40,6.0,8,0.398932
4,ARKQ,2026-04-17,call,117.0,0.90,2.05,5.0,5,0.396124
5,ARKQ,2026-04-17,call,118.0,0.75,1.70,15.0,18,0.388922
6,ARKQ,2026-04-17,call,119.0,0.20,1.90,1.0,2,0.444341
7,ARKQ,2026-04-17,call,120.0,0.30,1.20,3.0,12,0.385504
8,ARKQ,2026-04-17,call,121.0,0.10,1.05,1.0,5,0.391608
9,ARKQ,2026-04-17,call,122.0,0.05,0.80,2.0,6,0.378424


In [14]:
# Quick summary

summary = (
    options.groupby(["ticker", "contractType"])
    .agg(
        expirations   = ("expiration", "nunique"),
        contracts     = ("strike",     "count"),
        avg_iv        = ("impliedVolatility", "mean"),
        total_oi      = ("openInterest", "sum"),
    )
    .round(4)
)
print(summary)

                     expirations  contracts  avg_iv  total_oi
ticker contractType                                          
ARKQ   call                    5        135  0.4992      1957
       put                     5         90  0.3300      1490
BOTZ   call                    4         58  0.4989      4771
       put                     4         38  0.5088      6001
QQQ    call                   30       3773  0.3505   3400227
       put                    30       3397  0.3175   4727535
VOO    call                   16        841  0.3444     53899
       put                    16        813  0.3455     38500


In [15]:
# Save to disk

options.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV     → {OUTPUT_CSV}  ({options.shape[0]:,} rows)")

Saved CSV     → options_chain.csv  (9,145 rows)


# Section 3 — Spot Price Join

In [16]:
# Load saved data

options = pd.read_csv("options_chain.csv", parse_dates=["expiration"])
prices  = pd.read_csv("price_history.csv", parse_dates=["Date"])

print(f"Options rows : {len(options):,}")
print(f"Price rows   : {len(prices):,}")

Options rows : 9,145
Price rows   : 500


In [17]:
# Build spot price lookup table

# We want the most recent closing price for each ticker as of today (the snapshot date when data was collected).
# Since price history is sorted oldest -> newest, the last row per ticker is the most recent closing price.

spot = (
    prices.sort_values("Date")
    .groupby("ticker", as_index=False)
    .last()                          # most recent row per ticker
    [["ticker", "Date", "Close"]]
    .rename(columns={
        "Close": "spot_price",
        "Date" : "spot_date",
    })
)

print("\nSpot prices used for join:")
print(spot.to_string(index=False))


Spot prices used for join:
ticker  spot_date  spot_price
  ARKQ 2026-03-31  112.449997
  BOTZ 2026-03-31   33.220001
   QQQ 2026-03-31  577.179993
   VOO 2026-03-31  597.549988


In [18]:
# Join spot price onto options chain

# Simple left join on ticker — every option row gets the spot price of its underlying ETF on the collection date.

options_with_spot = options.merge(spot[["ticker", "spot_price"]], 
                                   on="ticker", 
                                   how="left")

print(f"\nRows before join : {len(options):,}")
print(f"Rows after join  : {len(options_with_spot):,}")


Rows before join : 9,145
Rows after join  : 9,145


In [19]:
# Verify no unexpected NaNs in spot_price

nan_count = options_with_spot["spot_price"].isna().sum()

if nan_count == 0:
    print("✓ No NaNs in spot_price — join successful.")
else:
    # Show which tickers failed to join
    problem_tickers = (
        options_with_spot[options_with_spot["spot_price"].isna()]
        ["ticker"]
        .unique()
    )
    print(f"⚠ {nan_count:,} NaNs found in spot_price.")
    print(f"  Affected tickers: {problem_tickers}")
    print("  Check that these tickers exist in price_history.csv.")


✓ No NaNs in spot_price — join successful.


In [20]:
# Compute moneyness

# Moneyness = strike / spot price
# = 1.0  → at the money  (strike == spot)
# > 1.0  → out of the money for calls / in the money for puts
# < 1.0  → in the money for calls  / out of the money for puts

options_with_spot["moneyness"] = (
    options_with_spot["strike"] / options_with_spot["spot_price"]
).round(4)

print("\nMoneyness sample:")
print(
    options_with_spot[["ticker", "expiration", "contractType", 
                        "strike", "spot_price", "moneyness"]]
    .head(10)
    .to_string(index=False)
)


Moneyness sample:
ticker expiration contractType  strike  spot_price  moneyness
  ARKQ 2026-04-17         call   105.0  112.449997     0.9337
  ARKQ 2026-04-17         call   110.0  112.449997     0.9782
  ARKQ 2026-04-17         call   115.0  112.449997     1.0227
  ARKQ 2026-04-17         call   116.0  112.449997     1.0316
  ARKQ 2026-04-17         call   117.0  112.449997     1.0405
  ARKQ 2026-04-17         call   118.0  112.449997     1.0494
  ARKQ 2026-04-17         call   119.0  112.449997     1.0582
  ARKQ 2026-04-17         call   120.0  112.449997     1.0671
  ARKQ 2026-04-17         call   121.0  112.449997     1.0760
  ARKQ 2026-04-17         call   122.0  112.449997     1.0849


In [21]:
# Quick sanity check

summary = (
    options_with_spot.groupby("ticker")
    .agg(
        contracts  = ("strike",     "count"),
        spot_price = ("spot_price", "first"),
        min_strike = ("strike",     "min"),
        max_strike = ("strike",     "max"),
        min_money  = ("moneyness",  "min"),
        max_money  = ("moneyness",  "max"),
    )
    .round(4)
)
print("\nSummary by ticker:")
print(summary)


Summary by ticker:
        contracts  spot_price  min_strike  max_strike  min_money  max_money
ticker                                                                     
ARKQ          225      112.45       55.00       195.0     0.4891     1.7341
BOTZ           96       33.22       20.00        50.0     0.6020     1.5051
QQQ          7170      577.18      174.78       950.0     0.3028     1.6459
VOO          1654      597.55      200.00       960.0     0.3347     1.6066


In [22]:
# Save merged dataset

OUTPUT_MERGED = "options_with_spot.csv"

options_with_spot.to_csv(OUTPUT_MERGED, index=False)
print(f"\nSaved → {OUTPUT_MERGED}  ({options_with_spot.shape[0]:,} rows, "
      f"{options_with_spot.shape[1]} columns)")



Saved → options_with_spot.csv  (9,145 rows, 11 columns)


In [23]:
# Reload check

reloaded = pd.read_csv(OUTPUT_MERGED, parse_dates=["expiration"])
assert reloaded.shape == options_with_spot.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, "
      f"{reloaded.shape[1]} columns.")
reloaded.head()

Reload check passed — 9,145 rows, 11 columns.


,ticker,expiration,contractType,strike,bid,ask,volume,openInterest,impliedVolatility,spot_price,moneyness
0,ARKQ,2026-04-17,call,105.0,7.30,9.70,16.0,1,0.549321,112.449997,0.9337
1,ARKQ,2026-04-17,call,110.0,3.60,5.50,12.0,6,0.434576,112.449997,0.9782
2,ARKQ,2026-04-17,call,115.0,1.65,2.65,5.0,3,0.387091,112.449997,1.0227
3,ARKQ,2026-04-17,call,116.0,1.35,2.40,6.0,8,0.398932,112.449997,1.0316
4,ARKQ,2026-04-17,call,117.0,0.90,2.05,5.0,5,0.396124,112.449997,1.0405


# Section 4 — Holdings

In [24]:
# Imports

import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [25]:
# Configuration

TICKERS     = ["VOO", "QQQ", "ARKQ", "BOTZ"]
OUTPUT_CSV  = "etf_holdings.csv"
DELAY       = 1.5   # seconds between requests — polite scraping

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def build_url(ticker: str) -> str:
    return f"https://stockanalysis.com/etf/{ticker.lower()}/holdings/"


In [26]:
# Scraper function

def scrape_holdings(ticker: str) -> pd.DataFrame:
    """
    Scrape top holdings from stockanalysis.com for a given ETF ticker.

    Returns a DataFrame with columns:
        holding_ticker, company_name, weight_pct, etf_ticker

    Raises ValueError if the page or table cannot be parsed.
    """
    url = build_url(ticker)

    # --- Fetch page ---
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise ValueError(f"HTTP request failed for {ticker}: {e}") from e

    soup = BeautifulSoup(resp.text, "lxml")

    # --- Find the holdings table ---
    table = soup.find("table")
    if table is None:
        raise ValueError(
            f"No table found for {ticker}. "
            "Page structure may have changed."
        )

    # --- Parse rows ---
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 4:
            continue

        # Column order: No. | Symbol | Name | % Weight | Shares
        symbol  = cells[1].get_text(strip=True)
        name    = cells[2].get_text(strip=True)
        weight  = cells[3].get_text(strip=True)

        # Clean weight — strip "%" and convert to float
        try:
            weight_float = float(weight.replace("%", "").strip())
        except ValueError:
            weight_float = float("nan")

        rows.append({
            "holding_ticker": symbol,
            "company_name"  : name,
            "weight_pct"    : weight_float,
            "etf_ticker"    : ticker.upper(),
        })

    if not rows:
        raise ValueError(f"Table found but no rows parsed for {ticker}.")

    return pd.DataFrame(rows)


In [27]:
# Fetch holdings for all tickers

frames = []

for i, ticker in enumerate(TICKERS):
    print(f"Scraping holdings: {ticker} ...", end=" ")
    try:
        df = scrape_holdings(ticker)
        frames.append(df)
        print(f"✓  {len(df)} holdings")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

    # Polite delay between requests (skip after last ticker)
    if i < len(TICKERS) - 1:
        time.sleep(DELAY)

if not frames:
    raise RuntimeError("No holdings data collected.")

holdings = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows: {len(holdings):,}")

Scraping holdings: VOO ... ✓  25 holdings
Scraping holdings: QQQ ... ✓  25 holdings
Scraping holdings: ARKQ ... ✓  25 holdings
Scraping holdings: BOTZ ... ✓  25 holdings

Total rows: 100


In [28]:
# Verify and preview

# Check for unexpected NaNs
nan_weights = holdings["weight_pct"].isna().sum()
if nan_weights == 0:
    print("✓ No NaNs in weight_pct")
else:
    print(f"⚠ {nan_weights} NaN weights — inspect rows below:")
    print(holdings[holdings["weight_pct"].isna()])

# Coverage check — confirm all 4 ETFs scraped
scraped = holdings["etf_ticker"].unique()
missing = set(TICKERS) - set(scraped)
if missing:
    print(f"⚠ Missing ETFs: {missing}")
else:
    print(f"✓ All ETFs present: {sorted(scraped)}")

# Preview top 5 holdings per ETF
print("\nTop 5 holdings per ETF:")
print(
    holdings.groupby("etf_ticker")
    .head(5)
    .to_string(index=False)
)

✓ No NaNs in weight_pct
✓ All ETFs present: ['ARKQ', 'BOTZ', 'QQQ', 'VOO']

Top 5 holdings per ETF:
holding_ticker                              company_name  weight_pct etf_ticker
          NVDA                        NVIDIA Corporation        7.31        VOO
          AAPL                                Apple Inc.        6.63        VOO
          MSFT                     Microsoft Corporation        4.96        VOO
          AMZN                          Amazon.com, Inc.        3.47        VOO
         GOOGL                             Alphabet Inc.        3.08        VOO
          NVDA                        NVIDIA Corporation        8.55        QQQ
          AAPL                                Apple Inc.        7.67        QQQ
          MSFT                     Microsoft Corporation        5.56        QQQ
          AMZN                          Amazon.com, Inc.        4.49        QQQ
          TSLA                               Tesla, Inc.        3.79        QQQ
          TSLA      

In [29]:
# Summary stats

summary = (
    holdings.groupby("etf_ticker")
    .agg(
        num_holdings    = ("holding_ticker", "count"),
        top10_weight    = ("weight_pct",     lambda x: x.head(10).sum()),
        max_weight      = ("weight_pct",     "max"),
    )
    .round(2)
)
print(summary)

            num_holdings  top10_weight  max_weight
etf_ticker                                        
ARKQ                  25         54.63       10.02
BOTZ                  25         57.76        8.19
QQQ                   25         46.27        8.55
VOO                   25         36.36        7.31


In [30]:
# Save to CSV

holdings.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}  ({holdings.shape[0]:,} rows, "
      f"{holdings.shape[1]} columns)")


Saved → etf_holdings.csv  (100 rows, 4 columns)


In [31]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV)
assert reloaded.shape == holdings.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows.")
reloaded.head()

Reload check passed — 100 rows.


,holding_ticker,company_name,weight_pct,etf_ticker
0,NVDA,NVIDIA Corporation,7.31,VOO
1,AAPL,Apple Inc.,6.63,VOO
2,MSFT,Microsoft Corporation,4.96,VOO
3,AMZN,"Amazon.com, Inc.",3.47,VOO
4,GOOGL,Alphabet Inc.,3.08,VOO


# Section 5 — Earnings